<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week4/week4_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Cell 1 — Week 4 Header & Library Imports
# Eye of Ra 👁️ — Week 4: Streamlit Dashboard & Technical Abstract

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Libraries loaded — Week 4 begins")
print("👁️ Eye of Ra — Dashboard & Submission")

# Git: Cell 1 — Week 4 header and imports

✅ Libraries loaded — Week 4 begins
👁️ Eye of Ra — Dashboard & Submission


In [5]:
# Cell 2 — Load dataset and scored contracts
df = pd.read_csv('/content/eye_of_ra_master_dataset_v3.csv')
scored = pd.read_csv('/content/eye_of_ra_scored_contracts_v1.csv')
model = joblib.load('/content/eye_of_ra_model_v1.pkl')
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')

print(f"✅ Master dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ Scored contracts: {scored.shape[0]} rows x {scored.shape[1]} columns")
print(f"✅ Model loaded: {type(model).__name__}")
print(f"✅ Features loaded: {len(feature_cols)} features")
print(f"\nTier breakdown:")
print(scored['tier'].value_counts())

# Git: Cell 2 — load all files for Week 4

✅ Master dataset: 262 rows x 21 columns
✅ Scored contracts: 262 rows x 10 columns
✅ Model loaded: XGBClassifier
✅ Features loaded: 12 features

Tier breakdown:
tier
🟢 MONITOR     240
🔴 ESCALATE     17
🟡 REVIEW        5
Name: count, dtype: int64


In [6]:
# Cell 3 — Install Streamlit and ngrok
!pip install streamlit pyngrok -q

from pyngrok import ngrok

print("✅ Streamlit installed")
print("✅ pyngrok installed")

# Git: Cell 3 — install Streamlit and ngrok

✅ Streamlit installed
✅ pyngrok installed


In [7]:
# Cell 4 — Write the Streamlit dashboard file

dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Page config
st.set_page_config(
    page_title="Eye of Ra 👁️",
    page_icon="👁️",
    layout="wide"
)

# Load data
@st.cache_data
def load_data():
    scored = pd.read_csv("/content/eye_of_ra_scored_contracts_v1.csv")
    return scored

scored = load_data()

# Clean up flags column for display
scored["flags"] = scored["flags"].fillna("[]")

# ── SIDEBAR ──────────────────────────────────────────────
st.sidebar.image("https://flagcdn.com/w80/gh.png", width=60)
st.sidebar.title("👁️ Eye of Ra")
st.sidebar.markdown("AI Procurement Fraud Detection for Ghana")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", [
    "📊 Overview",
    "🔴 Flagged Contracts",
    "🏛️ Entity Scorecards",
    "🔍 Contract Deep Dive"
])
st.sidebar.markdown("---")
st.sidebar.markdown("Built on Ghana PPA & Auditor-General data")
st.sidebar.markdown("Grounded in Public Procurement Act 663")

# ── PAGE 1: OVERVIEW ─────────────────────────────────────
if page == "📊 Overview":
    st.title("👁️ Eye of Ra — Procurement Fraud Detection")
    st.markdown("### Ghana AI Summit 2026 | Real Data. Real Flags. Real Accountability.")
    st.markdown("---")

    # Top metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Contracts Scanned", "262")
    col2.metric("Fraud Detected", "87%", "20 of 23 confirmed")
    col3.metric("🔴 Escalate", str(len(scored[scored["tier"] == "🔴 ESCALATE"])))
    col4.metric("False Positives", "0", "in ESCALATE tier")

    st.markdown("---")

    col1, col2 = st.columns(2)

    # Tier distribution pie
    with col1:
        tier_counts = scored["tier"].value_counts()
        fig = px.pie(
            values=tier_counts.values,
            names=tier_counts.index,
            title="Risk Tier Distribution — All 262 Contracts",
            color=tier_counts.index,
            color_discrete_map={
                "🔴 ESCALATE": "#d32f2f",
                "🟡 REVIEW": "#f9a825",
                "🟢 MONITOR": "#388e3c"
            }
        )
        st.plotly_chart(fig, use_container_width=True)

    # Fraud detection bar
    with col2:
        fig2 = go.Figure(go.Bar(
            x=["🔴 ESCALATE\\n(17 caught)", "🟡 REVIEW\\n(3 caught)", "🟢 MONITOR\\n(3 missed)"],
            y=[17, 3, 3],
            marker_color=["#d32f2f", "#f9a825", "#388e3c"],
            text=[17, 3, 3],
            textposition="outside"
        ))
        fig2.update_layout(
            title="23 Confirmed Fraud Contracts — Detection Breakdown",
            yaxis_title="Number of Contracts",
            showlegend=False
        )
        st.plotly_chart(fig2, use_container_width=True)

    # Score distribution
    st.markdown("### Score Distribution — Fraud vs Clean")
    fig3 = px.histogram(
        scored, x="composite_score", color="fraud_label",
        nbins=30,
        color_discrete_map={0: "#388e3c", 1: "#d32f2f"},
        labels={"fraud_label": "Fraud", "composite_score": "Composite Risk Score"},
        title="Risk Score Distribution"
    )
    fig3.add_vline(x=50, line_dash="dash", line_color="#d32f2f", annotation_text="ESCALATE threshold")
    fig3.add_vline(x=35, line_dash="dash", line_color="#f9a825", annotation_text="REVIEW threshold")
    st.plotly_chart(fig3, use_container_width=True)

# ── PAGE 2: FLAGGED CONTRACTS ─────────────────────────────
elif page == "🔴 Flagged Contracts":
    st.title("🔴 Flagged Contracts")
    st.markdown("Contracts in ESCALATE and REVIEW tiers — sorted by risk score.")
    st.markdown("---")

    tier_filter = st.multiselect(
        "Filter by tier:",
        options=["🔴 ESCALATE", "🟡 REVIEW", "🟢 MONITOR"],
        default=["🔴 ESCALATE", "🟡 REVIEW"]
    )

    filtered = scored[scored["tier"].isin(tier_filter)].sort_values("composite_score", ascending=False)

    st.markdown(f"**{len(filtered)} contracts shown**")
    st.dataframe(
        filtered[["entity", "supplier", "composite_score", "tier", "fraud_label"]].reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 3: ENTITY SCORECARDS ─────────────────────────────
elif page == "🏛️ Entity Scorecards":
    st.title("🏛️ Entity Integrity Scorecards")
    st.markdown("Search any procurement entity and see their full risk profile.")
    st.markdown("---")

    entity_list = sorted(scored["entity"].unique())
    selected_entity = st.selectbox("Select an entity:", entity_list)

    entity_contracts = scored[scored["entity"] == selected_entity]
    max_score = entity_contracts["composite_score"].max()
    fraud_count = entity_contracts["fraud_label"].sum()
    total = len(entity_contracts)
    escalate_count = len(entity_contracts[entity_contracts["tier"] == "🔴 ESCALATE"])

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Contracts", total)
    col2.metric("Max Risk Score", f"{max_score:.1f}")
    col3.metric("Confirmed Fraud", int(fraud_count))
    col4.metric("Escalated", escalate_count)

    st.markdown("---")
    st.markdown("### All contracts for this entity:")
    st.dataframe(
        entity_contracts[["supplier", "composite_score", "tier", "fraud_label"]].sort_values("composite_score", ascending=False).reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 4: CONTRACT DEEP DIVE ────────────────────────────
elif page == "🔍 Contract Deep Dive":
    st.title("🔍 Contract Deep Dive")
    st.markdown("Select any contract for a full risk breakdown.")
    st.markdown("---")

    contract_idx = st.selectbox(
        "Select contract index:",
        options=scored.index.tolist(),
        format_func=lambda i: f"#{i} — {scored.loc[i, 'entity'][:50]}"
    )

    row = scored.loc[contract_idx]

    col1, col2 = st.columns(2)
    with col1:
        st.markdown(f"**Entity:** {row['entity']}")
        st.markdown(f"**Supplier:** {row['supplier']}")
        st.markdown(f"**Composite Score:** {row['composite_score']}")
        st.markdown(f"**Tier:** {row['tier']}")
        st.markdown(f"**Fraud Confirmed:** {'✅ Yes' if row['fraud_label'] == 1 else '⬜ No'}")

    with col2:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=row["composite_score"],
            title={"text": "Risk Score"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"color": "#d32f2f" if row["composite_score"] >= 50 else "#f9a825" if row["composite_score"] >= 35 else "#388e3c"},
                "steps": [
                    {"range": [0, 35], "color": "#e8f5e9"},
                    {"range": [35, 50], "color": "#fff9c4"},
                    {"range": [50, 100], "color": "#ffebee"}
                ],
                "threshold": {"line": {"color": "black", "width": 4}, "thickness": 0.75, "value": row["composite_score"]}
            }
        ))
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    st.markdown("### Flags raised:")
    flags = eval(row["flags"]) if isinstance(row["flags"], str) else row["flags"]
    if flags:
        for f in flags:
            st.markdown(f"• {f}")
    else:
        st.markdown("No flags raised.")

    st.markdown("### Legal citations:")
    citations = eval(row["legal_citations"]) if isinstance(row["legal_citations"], str) else row["legal_citations"]
    if citations:
        for c in citations:
            st.markdown(f"• {c}")

    st.markdown("### Top SHAP drivers:")
    shap_items = eval(row["shap_top3"]) if isinstance(row["shap_top3"], str) else row["shap_top3"]
    if shap_items:
        for s in shap_items:
            st.markdown(f"• {s}")
'''

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ Dashboard file written: /content/app.py")

# Git: Cell 4 — Streamlit dashboard file written

✅ Dashboard file written: /content/app.py


In [7]:
# Cell 5 — Launch dashboard with localtunnel (no account needed)
import subprocess
import threading
import time

# Install localtunnel
subprocess.run(["npm", "install", "-g", "localtunnel"], capture_output=True)

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to boot
time.sleep(6)

# Start localtunnel
tunnel = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
output = tunnel.stdout.readline().decode("utf-8").strip()

print("=" * 55)
print("👁️  EYE OF RA DASHBOARD IS LIVE")
print("=" * 55)
print(f"\n🌐 Open this URL in your browser:")
print(f"   {output}")
print(f"\nIf it asks for a password, go to https://loca.lt/mytunnelpassword")
print(f"Copy the password shown there and paste it into the tunnel page.")
print(f"\nKeep this cell running — closing it kills the dashboard.")

# Git: Cell 5 — dashboard launched via localtunnel

👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL in your browser:
   your url is: https://some-banks-buy.loca.lt

If it asks for a password, go to https://loca.lt/mytunnelpassword
Copy the password shown there and paste it into the tunnel page.

Keep this cell running — closing it kills the dashboard.


In [8]:
# Cell 5 — Launch dashboard with cloudflared
import subprocess
import threading
import time

# Install cloudflared
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("✅ cloudflared installed")

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to boot
time.sleep(6)
print("✅ Streamlit running")

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Extract the URL from output
print("⏳ Starting tunnel...")
for _ in range(20):
    line = tunnel.stderr.readline().decode("utf-8")
    if "trycloudflare.com" in line:
        url = [x for x in line.split() if "trycloudflare.com" in x][0]
        print("=" * 55)
        print("👁️  EYE OF RA DASHBOARD IS LIVE")
        print("=" * 55)
        print(f"\n🌐 Open this URL:")
        print(f"   {url}")
        print("\nNo password needed.")
        break
    time.sleep(1)

# Git: Cell 5 — dashboard launched via cloudflared

CalledProcessError: Command '['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared']' returned non-zero exit status 1.

In [9]:
# Cell 5 — Launch dashboard with cloudflared (fixed)
import subprocess
import threading
import time
import shutil

# Confirm streamlit path
streamlit_path = shutil.which("streamlit")
print(f"✅ Streamlit found at: {streamlit_path}")

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        streamlit_path, "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(6)
print("✅ Streamlit running")

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Extract URL
print("⏳ Starting tunnel...")
for _ in range(30):
    line = tunnel.stderr.readline().decode("utf-8")
    if "trycloudflare.com" in line:
        # Extract just the URL
        parts = line.split()
        for part in parts:
            if "trycloudflare.com" in part:
                url = part.strip()
                print("=" * 55)
                print("👁️  EYE OF RA DASHBOARD IS LIVE")
                print("=" * 55)
                print(f"\n🌐 Open this URL:")
                print(f"   {url}")
                print("\nNo password needed.")
                break
        break
    time.sleep(1)

# Git: Cell 5 — dashboard launched, streamlit path fixed

✅ Streamlit found at: /usr/local/bin/streamlit
✅ Streamlit running
⏳ Starting tunnel...
👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL:
   trycloudflare.com...

No password needed.


In [10]:
# Cell 5 — Reinstall Streamlit and launch with cloudflared
import subprocess
import threading
import time

# Reinstall streamlit
print("⏳ Installing Streamlit...")
subprocess.run(["pip", "install", "streamlit", "-q"])

# Confirm path
result = subprocess.run(["which", "streamlit"], capture_output=True, text=True)
streamlit_path = result.stdout.strip()
print(f"✅ Streamlit found at: {streamlit_path}")

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        streamlit_path, "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(8)
print("✅ Streamlit running")

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Extract URL
print("⏳ Starting tunnel...")
for _ in range(30):
    line = tunnel.stderr.readline().decode("utf-8")
    if "trycloudflare.com" in line:
        parts = line.split()
        for part in parts:
            if "trycloudflare.com" in part:
                url = part.strip()
                print("=" * 55)
                print("👁️  EYE OF RA DASHBOARD IS LIVE")
                print("=" * 55)
                print(f"\n🌐 Open this URL:")
                print(f"   {url}")
                print("\nNo password needed.")
                break
        break
    time.sleep(1)

# Git: Cell 5 — streamlit reinstalled, dashboard launched

⏳ Installing Streamlit...
✅ Streamlit found at: /usr/local/bin/streamlit
✅ Streamlit running
⏳ Starting tunnel...
👁️  EYE OF RA DASHBOARD IS LIVE

🌐 Open this URL:
   trycloudflare.com...

No password needed.


In [11]:
# Cell 5 — Fix URL extraction
import subprocess
import threading
import time

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        "/usr/local/bin/streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(8)
print("✅ Streamlit running")

# Start cloudflared and print ALL stderr lines so we can see the full URL
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Reading tunnel output...")
for _ in range(40):
    line = tunnel.stderr.readline().decode("utf-8").strip()
    if line:
        print(line)  # print every line so we can see exactly what comes out
    if "trycloudflare.com" in line:
        break
    time.sleep(1)

# Git: Cell 5 — debug URL extraction

✅ Streamlit running
⏳ Reading tunnel output...
2026-06-06T06:17:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-06T06:17:01Z INF Requesting new quick Tunnel on trycloudflare.com...


In [12]:
# Cell 5b — Keep reading tunnel output until URL appears
print("⏳ Waiting for URL...")
for _ in range(60):
    line = tunnel.stderr.readline().decode("utf-8").strip()
    if line:
        print(line)
    if "trycloudflare.com" in line and "http" in line:
        break
    time.sleep(1)

⏳ Waiting for URL...
2026-06-06T06:17:05Z INF +--------------------------------------------------------------------------------------------+
2026-06-06T06:17:05Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-06T06:17:05Z INF |  https://lawrence-much-protective-including.trycloudflare.com                              |


In [13]:
# Cell 6 — Standardize entity names in master dataset
# Fix duplicate/inconsistent entity names before re-scoring

entity_mapping = {
    # BOST
    "Bulk Oil Storage and Transportation": "Bulk Oil Storage and Transportation Company Limited (BOST)",
    "Bulk Oil Storage Transportation Company Ltd. (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",
    "Bulk Oil Storage and Transportation company Limited (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",
    "Bulk Oil Storage and Transportation Company Ltd. (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",

    # Bank of Ghana
    "Bank of Ghana": "Bank of Ghana (BoG)",

    # Accra Technical University
    "Accra Technical University": "Accra Technical University (ATU)",

    # Ghana Gas
    "Ghana National Gas Company": "Ghana National Gas Company (Ghana Gas)",
    "Ghana Gas": "Ghana National Gas Company (Ghana Gas)",
    "Ghana National Gas Company (GNGC)": "Ghana National Gas Company (Ghana Gas)",

    # GPHA
    "Ghana Ports and Harbours Authority": "Ghana Ports and Harbours Authority (GPHA)",

    # COCOBOD
    "Ghana Cocoa Board": "Ghana Cocoa Board (COCOBOD)",

    # Ministry of Finance
    "Ministry of Finance": "Ministry of Finance (MoF)",

    # Korle Bu
    "Korle-Bu Teaching Hospital": "Korle Bu Teaching Hospital (KBTH)",
    "Korle Bu Teaching Hospital": "Korle Bu Teaching Hospital (KBTH)",
}

# Apply mapping
df['entity'] = df['entity'].replace(entity_mapping)

# Check remaining duplicates
print("=" * 55)
print("👁️  ENTITY NAME STANDARDIZATION")
print("=" * 55)
print(f"Unique entities before: 133")
print(f"Unique entities after:  {df['entity'].nunique()}")
print(f"\nAll unique entity names:")
for e in sorted(df['entity'].unique()):
    print(f"  {e}")

# Git: Cell 6 — entity name standardization

👁️  ENTITY NAME STANDARDIZATION
Unique entities before: 133
Unique entities after:  123

All unique entity names:
  Accra Technical University (ATU)
  Bank of Ghana (BoG)
  Bulk Oil Storage and Transportatio n Company Limited (BOST)
  Bulk Oil Storage and Transportation Company Limited (BOST)
  Cape Coast Technical University (CCTU)
  Centre for Plant Medicine Research
  Coastal Development Authority (CODA)
  Controller and Accountant General’s Department (CAGD)
  Driver and Vehicle Licensing Authority
  Driver and Vehicle Licensing Authority (DVLA)
  Electoral Commission
  Electoral Commission (EC)
  GIFEC
  GNPC
  Ghana Airport Company Ltd
  Ghana Airport Company Ltd. (GACL)
  Ghana Airports Company Limited (GACL)
  Ghana Cocoa Board (COCOBOD)
  Ghana Cocoa Board (Cocoa Board)
  Ghana College of Physicians and Surgeons
  Ghana Commodity Exchange (GCX)
  Ghana Education Services (GES)
  Ghana Export Promotion Authority (GEPA)
  Ghana Export Promotion Authority (GEPA))
  Ghana Geologic

In [14]:
# Cell 7 — Fix all remaining entity name duplicates

entity_mapping_2 = {
    # BOST (typo with space)
    "Bulk Oil Storage and Transportatio n Company Limited (BOST)": "Bulk Oil Storage and Transportation Company Limited (BOST)",

    # DVLA
    "Driver and Vehicle Licensing Authority": "Driver and Vehicle Licensing Authority (DVLA)",

    # Electoral Commission
    "Electoral Commission": "Electoral Commission (EC)",

    # GIFEC
    "GIFEC": "Ghana Investment Fund for Electronic Communications (GIFEC)",

    # Ghana Airport
    "Ghana Airport Company Ltd": "Ghana Airports Company Limited (GACL)",
    "Ghana Airport Company Ltd. (GACL)": "Ghana Airports Company Limited (GACL)",

    # Ghana Cocoa Board
    "Ghana Cocoa Board (Cocoa Board)": "Ghana Cocoa Board (COCOBOD)",

    # Ghana Export Promotion Authority (typo extra bracket)
    "Ghana Export Promotion Authority (GEPA))": "Ghana Export Promotion Authority (GEPA)",

    # Ghana Gas / GNPC confusion
    "Ghana National Gas Company (GNPC)": "Ghana National Gas Company (Ghana Gas)",
    "Ghana National Gas Company Ltd": "Ghana National Gas Company (Ghana Gas)",
    "GNPC": "Ghana National Petroleum Corporation (GNPC)",

    # Ghana Maritime Authority
    "Ghana Maritime Authority": "Ghana Maritime Authority (GMA)",

    # Ghana Railway
    "Ghana Railway Authority": "Ghana Railways Development Authority",
    "Ghana Railway Company Ltd": "Ghana Railways Development Authority",
    "Ghana Railways Company Ltd": "Ghana Railways Development Authority",
    "Ministry of Railways Development": "Ministry of Railway Development (MoRD)",

    # Ghana Statistical Service
    "Ghana Statistical Service": "Ghana Statistical Service (GSS)",

    # Ministry of Education
    "Ministry of Education": "Ministry of Education (MoE)",

    # Ministry of Health
    "Ministry of Health": "Ministry of Health (MoH)",
    "Ministry of Health (MOH)": "Ministry of Health (MoH)",

    # Ministry of Local Government
    "Ministry of Local Government Decentralization and Rural Development (MLGDRD)": "Ministry of Local Government, Decentralization and Rural Development (MLGDRD)",
    "Ministry of Local Government and Rural Development (MLRD)": "Ministry of Local Government, Decentralization and Rural Development (MLGDRD)",
    "Ministry of Local Government, Decentralization and Rural Development (MLGDRD),": "Ministry of Local Government, Decentralization and Rural Development (MLGDRD)",

    # Ministry of Youth and Sports
    "Ministry of Youth and Sports (MoYS)": "Ministry of Youth and Sports (MOYS)",

    # NHIA
    "NHIA": "National Health Insurance Authority (NHIA)",
    "National Health Insurance Authority": "National Health Insurance Authority (NHIA)",
    "National Health Insurance Authority (NHIS)": "National Health Insurance Authority (NHIA)",

    # National Cardiothoracic Centre (typo with space)
    "National Cardiothoraci c Centre (NCTC)": "National Cardiothoracic Centre (NCTC)",

    # National Centre for Radiotherapy
    "National Centre for Radiotherapy and": "National Centre for Radiotherapy and Nuclear Medicine (NCRNM)",
    "Nuclear Medicine (NCRNM) of Korle Bu Teaching Hospital (KBTH)": "National Centre for Radiotherapy and Nuclear Medicine (NCRNM)",
    "National Centre for Radiotherapy and Nuclear Medicine (NCRNM) of Korle Bu Teaching Hospital (KBTH)": "National Centre for Radiotherapy and Nuclear Medicine (NCRNM)",

    # National Communications Authority (typo with space)
    "National Communicati ons Authority (NCA)": "National Communications Authority (NCA)",

    # National Petroleum Authority
    "National Petroleum Authority": "National Petroleum Authority (NPA)",

    # Office of Attorney General (typo with space)
    "Office of the Attorney- General and Ministry of Justice": "Office of the Attorney-General and Ministry of Justice",

    # PSC Tema Shipyard
    "PSC Tema Shipyard": "PSC Tema Shipyard Limited (Tema Shipyard)",
    "PSC Tema Shipyard Limited": "PSC Tema Shipyard Limited (Tema Shipyard)",

    # Scholarship Secretariat
    "Scholarship Secretariat": "Scholarship Secretariat (SLA)",

    # Volta River Authority
    "Volta River Authority": "Volta River Authority (VRA)",
}

df['entity'] = df['entity'].replace(entity_mapping_2)

print("=" * 55)
print("👁️  ENTITY NAMES — FINAL CLEAN")
print("=" * 55)
print(f"Unique entities now: {df['entity'].nunique()}")
print(f"\nAll unique entity names:")
for e in sorted(df['entity'].unique()):
    print(f"  {e}")

# Git: "Cell 7 — all entity name duplicates fixed"

👁️  ENTITY NAMES — FINAL CLEAN
Unique entities now: 87

All unique entity names:
  Accra Technical University (ATU)
  Bank of Ghana (BoG)
  Bulk Oil Storage and Transportation Company Limited (BOST)
  Cape Coast Technical University (CCTU)
  Centre for Plant Medicine Research
  Coastal Development Authority (CODA)
  Controller and Accountant General’s Department (CAGD)
  Driver and Vehicle Licensing Authority (DVLA)
  Electoral Commission (EC)
  Ghana Airports Company Limited (GACL)
  Ghana Cocoa Board (COCOBOD)
  Ghana College of Physicians and Surgeons
  Ghana Commodity Exchange (GCX)
  Ghana Education Services (GES)
  Ghana Export Promotion Authority (GEPA)
  Ghana Geological Survey Authority (GGSA)
  Ghana Health Services (GHS)
  Ghana Investment Fund for Electronic Communications (GIFEC)
  Ghana Maritime Authority (GMA)
  Ghana National Gas Company (Ghana Gas)
  Ghana National Petroleum Corporation (GNPC)
  Ghana Police Service (GPS)
  Ghana Ports and Harbours Authority (GPHA)
  G

In [15]:
# Cell 8 — Save clean dataset and re-run full scoring pipeline

# Save clean master dataset
df.to_csv('/content/eye_of_ra_master_dataset_v4.csv', index=False)
print("✅ Clean dataset saved: eye_of_ra_master_dataset_v4.csv")

# Re-run all 5 engines on clean data
# ── ENGINE 1 ──
def bid_manipulation_score(row, df):
    score = 0
    flags = []
    if row['is_sole_source'] == 1:
        score += 30
        flags.append("Sole source procurement used (S.38-41 Act 663)")
    if row['is_variation'] == 1 and row['audit_flagged'] == 1:
        score += 25
        flags.append("Variation issued on audit-flagged contract (S.59 Act 663)")
    if row['award_concentration'] > 0.3:
        score += 20
        flags.append(f"Supplier holds {row['award_concentration']*100:.1f}% of entity awards — concentration risk")
    if row['method_abuse_score'] > 0.5:
        score += 15
        flags.append("Procurement method abuse pattern detected (S.40-41 Act 663)")
    if row['variation_abuse_rate'] > 0.3:
        score += 10
        flags.append(f"Entity variation abuse rate: {row['variation_abuse_rate']*100:.1f}%")
    return min(score, 100), flags

# ── ENGINE 2 ──
def build_supplier_profiles(df):
    profiles = {}
    for supplier in df['supplier'].unique():
        s = df[df['supplier'] == supplier]
        total = len(s)
        score = 0
        flags = []
        if s['fraud_label'].sum() > 0:
            score += 40
            flags.append(f"Supplier linked to {int(s['fraud_label'].sum())} confirmed fraud case(s)")
        if s['audit_flagged'].sum() > 0:
            score += 20
            flags.append(f"Supplier appears in {int(s['audit_flagged'].sum())} Auditor-General finding(s)")
        ss_rate = s['is_sole_source'].sum() / total if total > 0 else 0
        if ss_rate > 0.5:
            score += 20
            flags.append(f"{int(ss_rate*100)}% of awards were sole source — bypasses competition (S.38-41)")
        var_rate = s['is_variation'].sum() / total if total > 0 else 0
        if var_rate > 0.4:
            score += 15
            flags.append(f"{int(var_rate*100)}% of contracts have variations — inflation risk")
        if total >= 3 and s['entity'].nunique() == 1:
            score += 5
            flags.append(f"All {total} contracts with a single entity — capture risk")
        profiles[supplier] = {'engine2_score': min(score, 100), 'engine2_flags': flags}
    return profiles

# ── ENGINE 3 ──
def build_entity_variation_profiles(df):
    profiles = {}
    for entity in df['entity'].unique():
        e = df[df['entity'] == entity]
        total = len(e)
        var_rate = e['is_variation'].sum() / total if total > 0 else 0
        audit_flagged_var = e[(e['is_variation']==1) & (e['audit_flagged']==1)].shape[0]
        fraud_var = e[(e['is_variation']==1) & (e['fraud_label']==1)].shape[0]
        score = 0
        flags = []
        if var_rate > 0.4:
            score += 35
            flags.append(f"{int(var_rate*100)}% of contracts are variations — systematic inflation pattern (S.87 Act 663)")
        elif var_rate > 0.2:
            score += 15
            flags.append(f"{int(var_rate*100)}% variation rate — elevated above normal threshold")
        if audit_flagged_var > 0:
            score += 30
            flags.append(f"{audit_flagged_var} variation(s) independently flagged by Auditor-General")
        if fraud_var > 0:
            score += 35
            flags.append(f"{fraud_var} variation contract(s) confirmed fraudulent")
        profiles[entity] = {'engine3_score': min(score, 100), 'engine3_flags': flags}
    return profiles

# ── ENGINE 4 ──
def build_sole_source_profiles(df):
    profiles = {}
    for entity in df['entity'].unique():
        e = df[df['entity'] == entity]
        total = len(e)
        ss_rate = e['is_sole_source'].sum() / total if total > 0 else 0
        audit_ss = e[(e['is_sole_source']==1) & (e['audit_flagged']==1)].shape[0]
        fraud_ss = e[(e['is_sole_source']==1) & (e['fraud_label']==1)].shape[0]
        method_abuse = e['method_abuse_score'].mean()
        score = 0
        flags = []
        if ss_rate > 0.5:
            score += 40
            flags.append(f"{int(ss_rate*100)}% of contracts are sole source — systematic bypass of competition (S.40-41 Act 663)")
        elif ss_rate > 0.25:
            score += 20
            flags.append(f"{int(ss_rate*100)}% sole source rate — above acceptable threshold (S.40-41 Act 663)")
        if audit_ss > 0:
            score += 30
            flags.append(f"{audit_ss} sole source contract(s) flagged by Auditor-General")
        if fraud_ss > 0:
            score += 30
            flags.append(f"{fraud_ss} sole source contract(s) confirmed fraudulent")
        if method_abuse > 0.4:
            score += 15
            flags.append(f"Entity method abuse score: {method_abuse:.2f} — pattern of procurement rule circumvention")
        profiles[entity] = {'engine4_score': min(score, 100), 'engine4_flags': flags}
    return profiles

# ── ENGINE 5 ──
def build_entity_risk_profiles(df):
    profiles = {}
    for entity in df['entity'].unique():
        e = df[df['entity'] == entity]
        total = len(e)
        fraud_count = e['fraud_label'].sum()
        fraud_rate = fraud_count / total if total > 0 else 0
        audit_rate = e['audit_flagged'].sum() / total if total > 0 else 0
        avg_supplier_risk = e['supplier_risk_score'].mean()
        repeat_supplier_rate = e['repeat_supplier'].mean()
        avg_audit_intensity = e['audit_intensity'].mean()
        score = 0
        flags = []
        if fraud_count > 0:
            score += 35
            flags.append(f"{int(fraud_count)} of {total} contracts confirmed fraudulent ({int(fraud_rate*100)}% fraud rate)")
        if audit_rate > 0.3:
            score += 25
            flags.append(f"{int(audit_rate*100)}% of contracts flagged by Auditor-General — systemic oversight failure (S.2 Act 663)")
        elif e['audit_flagged'].sum() > 0:
            score += 10
            flags.append(f"{int(e['audit_flagged'].sum())} contract(s) flagged by Auditor-General")
        if avg_supplier_risk > 0.6:
            score += 20
            flags.append(f"Average supplier risk score: {avg_supplier_risk:.2f} — entity consistently awards to high-risk suppliers")
        if repeat_supplier_rate > 0.7:
            score += 15
            flags.append(f"{int(repeat_supplier_rate*100)}% repeat supplier rate — limited supplier diversity (S.3 Act 663)")
        if avg_audit_intensity > 0.5:
            score += 5
            flags.append(f"Audit intensity score: {avg_audit_intensity:.2f} — entity under sustained scrutiny")
        profiles[entity] = {'engine5_score': min(score, 100), 'engine5_flags': flags}
    return profiles

# ── RUN ALL ENGINES ──
print("⏳ Running all 5 engines on clean data...")

e1 = df.apply(lambda row: bid_manipulation_score(row, df), axis=1)
df['engine1_score'] = e1.apply(lambda x: x[0])
df['engine1_flags'] = e1.apply(lambda x: x[1])

sp = build_supplier_profiles(df)
df['engine2_score'] = df['supplier'].map(lambda s: sp[s]['engine2_score'])
df['engine2_flags'] = df['supplier'].map(lambda s: sp[s]['engine2_flags'])

evp = build_entity_variation_profiles(df)
df['engine3_score'] = df['entity'].map(lambda e: evp[e]['engine3_score'])
df['engine3_flags'] = df['entity'].map(lambda e: evp[e]['engine3_flags'])

esp = build_sole_source_profiles(df)
df['engine4_score'] = df['entity'].map(lambda e: esp[e]['engine4_score'])
df['engine4_flags'] = df['entity'].map(lambda e: esp[e]['engine4_flags'])

erp = build_entity_risk_profiles(df)
df['engine5_score'] = df['entity'].map(lambda e: erp[e]['engine5_score'])
df['engine5_flags'] = df['entity'].map(lambda e: erp[e]['engine5_flags'])

print("✅ All 5 engines complete")

# ── COMPOSITE SCORE ──
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')
df['ml_score'] = model.predict_proba(df[feature_cols])[:, 1] * 100
df['composite_score'] = (
    df['ml_score'] * 0.40 +
    df['engine1_score'] * 0.15 +
    df['engine2_score'] * 0.15 +
    df['engine3_score'] * 0.10 +
    df['engine4_score'] * 0.10 +
    df['engine5_score'] * 0.10
).round(1).clip(upper=100)

def assign_tier(score):
    if score >= 50:
        return "🔴 ESCALATE"
    elif score >= 35:
        return "🟡 REVIEW"
    else:
        return "🟢 MONITOR"

df['tier'] = df['composite_score'].apply(assign_tier)

# ── SUMMARY ──
fraud = df[df['fraud_label'] == 1]
escalate = fraud[fraud['tier'] == '🔴 ESCALATE']
review = fraud[fraud['tier'] == '🟡 REVIEW']
monitor = fraud[fraud['tier'] == '🟢 MONITOR']

print("=" * 55)
print("👁️  EYE OF RA — CLEAN DATA RESCORE")
print("=" * 55)
print(f"Contracts scored:      {len(df)}")
print(f"Unique entities:       {df['entity'].nunique()}")
print(f"\nFraud Detection:")
print(f"  🔴 ESCALATE: {len(escalate)} of 23 ({len(escalate)/23*100:.0f}%)")
print(f"  🟡 REVIEW:   {len(review)} of 23 ({len(review)/23*100:.0f}%)")
print(f"  🟢 MONITOR:  {len(monitor)} of 23 — missed")
print(f"  Total caught: {len(escalate)+len(review)} of 23 ({(len(escalate)+len(review))/23*100:.0f}%)")
print(f"\nFalse positives in ESCALATE: {len(df[(df['fraud_label']==0) & (df['tier']=='🔴 ESCALATE')])}")

# Save
df.to_csv('/content/eye_of_ra_scored_contracts_v2.csv', index=False)
print(f"\n✅ Saved: eye_of_ra_scored_contracts_v2.csv")

# Git: Cell 8 — clean data rescored, v2 saved

✅ Clean dataset saved: eye_of_ra_master_dataset_v4.csv
⏳ Running all 5 engines on clean data...
✅ All 5 engines complete
👁️  EYE OF RA — CLEAN DATA RESCORE
Contracts scored:      262
Unique entities:       87

Fraud Detection:
  🔴 ESCALATE: 18 of 23 (78%)
  🟡 REVIEW:   2 of 23 (9%)
  🟢 MONITOR:  3 of 23 — missed
  Total caught: 20 of 23 (87%)

False positives in ESCALATE: 0

✅ Saved: eye_of_ra_scored_contracts_v2.csv


In [16]:
# Cell 9 — Update dashboard to use clean v2 files

dashboard_code = open("/content/app.py").read()
dashboard_code = dashboard_code.replace(
    "eye_of_ra_scored_contracts_v1.csv",
    "eye_of_ra_scored_contracts_v2.csv"
)

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ Dashboard updated to use scored_contracts_v2.csv")
print("⏳ Restart the Streamlit server by rerunning Cell 5 to see changes.")

# Git: Cell 9 — dashboard updated to v2 scored contracts

✅ Dashboard updated to use scored_contracts_v2.csv
⏳ Restart the Streamlit server by rerunning Cell 5 to see changes.
